# EfficientNet Transfer Learning — Flower Classification (PyTorch)

**Model:** EfficientNet-B0 / B1 / B2 / B3 / B4 (pretrained on ImageNet)  
**Dataset:** Oxford Flowers 102 (102 flower species)  
**Focus:** Compound scaling — depth, width, resolution scale together

**EfficientNet Key Innovation — Compound Scaling:**
```
Compound Scaling Method:
  depth   = α^φ     (α = 1.2)   → more layers
  width   = β^φ     (β = 1.1)   → wider channels
  resolution = γ^φ  (γ = 1.15)  → higher input resolution
  
  Constraint: α · β² · γ² ≈ 2   (≈2× FLOPS increase per φ step)

  B0: φ=0 (baseline)  → 224×224, 5.3M params
  B1: φ=0.5            → 240×240, 7.8M params
  B2: φ=1.0            → 260×260, 9.2M params
  B3: φ=1.5            → 300×300, 12M params
  B4: φ=2.0            → 380×380, 19M params
  ...
  B7: φ=3.5            → 600×600, 66M params
```

**Architecture (EfficientNet-B0):**
```
Input (3 × 224 × 224)
    ↓
Stem: Conv2d(3→32, 3×3, stride=2) + BN + SiLU
    ↓
MBConv Blocks ×16 (7 stages)
  ├── MBConv1, k3×3, channels 32→16    (×1)
  ├── MBConv6, k3×3, channels 16→24    (×2)
  ├── MBConv6, k5×5, channels 24→40    (×2)
  ├── MBConv6, k3×3, channels 40→80    (×3)
  ├── MBConv6, k5×5, channels 80→112   (×3)
  ├── MBConv6, k5×5, channels 112→192  (×4)
  └── MBConv6, k3×3, channels 192→320  (×1)
    ↓
Head: Conv2d(320→1280) + BN + SiLU + AdaptiveAvgPool + Dropout
    ↓
Classifier: FC(1280 → num_classes)
```

> GPU accelerator ဖွင့်ပြီး run ပါ: `Settings → Accelerator → GPU`

In [ ]:
# --- 1. Install Dependencies ---
# !pip install -q torch torchvision datasets scikit-learn seaborn tqdm

In [ ]:
# --- 2. Imports & Seed ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import time
from PIL import Image
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ All imports successful.")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- 3. Device & Hyperparameters ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Hyperparameters ──
IMG_SIZE = 224          # EfficientNet-B0 default (B1→240, B2→260, B3→300, B4→380)
BATCH_SIZE = 32
NUM_EPOCHS = 20         # Phase 2 epochs
LEARNING_RATE = 1e-3    # Phase 1 LR (classifier only)
FINE_TUNE_LR = 1e-5     # Phase 2 base LR
FREEZE_EPOCHS = 5       # Phase 1 duration
NUM_CLASSES = 102       # Oxford Flowers 102

# EfficientNet variant — 'b0', 'b1', 'b2', 'b3', 'b4'
VARIANT = 'b0'  # Change to 'b1'..'b4' for larger models

# ImageNet normalization stats
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# EfficientNet compound scaling coefficients
# depth = α^φ, width = β^φ, resolution = γ^φ
# α = 1.2, β = 1.1, γ = 1.15, subject to α · β² · γ² ≈ 2
EFFICIENTNET_CONFIG = {
    'b0': {'resolution': 224, 'width_mult': 1.0, 'depth_mult': 1.0, 'dropout': 0.2},
    'b1': {'resolution': 240, 'width_mult': 1.0, 'depth_mult': 1.1, 'dropout': 0.2},
    'b2': {'resolution': 260, 'width_mult': 1.1, 'depth_mult': 1.2, 'dropout': 0.3},
    'b3': {'resolution': 300, 'width_mult': 1.2, 'depth_mult': 1.4, 'dropout': 0.3},
    'b4': {'resolution': 380, 'width_mult': 1.4, 'depth_mult': 1.8, 'dropout': 0.4},
}

# Update IMG_SIZE based on variant
IMG_SIZE = EFFICIENTNET_CONFIG[VARIANT]['resolution']

print(f"\n📐 EfficientNet-{VARIANT.upper()} Configuration:")
print(f"   Input resolution: {IMG_SIZE}×{IMG_SIZE}")
print(f"   Width multiplier: {EFFICIENTNET_CONFIG[VARIANT]['width_mult']}")
print(f"   Depth multiplier: {EFFICIENTNET_CONFIG[VARIANT]['depth_mult']}")
print(f"   Dropout rate:     {EFFICIENTNET_CONFIG[VARIANT]['dropout']}")
print(f"   Num classes:      {NUM_CLASSES}")
print(f"\n📖 Compound Scaling: depth=α^φ, width=β^φ, resolution=γ^φ")
print(f"   α=1.2, β=1.1, γ=1.15 → α·β²·γ² ≈ {1.2 * 1.1**2 * 1.15**2:.3f} ≈ 2")

## 📦 Dataset — Oxford Flowers 102

Oxford Flowers 102 dataset ကို HuggingFace `datasets` library နဲ့ download လုပ်ပါမယ်။  
- 102 flower species, ~8,000 images total  
- Fine-grained classification → EfficientNet ရဲ့ compound scaling ကြောင့် ပိုကောင်းတဲ့ features extract လုပ်နိုင်

In [ ]:
# --- 4. Download Oxford Flowers 102 ---
from datasets import load_dataset

print("⏳ Downloading Oxford Flowers 102...")
hf_dataset = load_dataset("nelorth/oxford-flowers")
print(f"✅ Download complete!")
print(f"Train: {len(hf_dataset['train'])} | Test: {len(hf_dataset['test'])}")

# Get class names (flower species)
FLOWER_NAMES = hf_dataset['train'].features['label'].names
print(f"Classes: {NUM_CLASSES}")
print(f"Sample classes: {FLOWER_NAMES[:10]}...")

In [ ]:
# --- 5. Custom Dataset Class ---
class FlowersDataset(Dataset):
    """Wraps HuggingFace dataset for PyTorch DataLoader."""
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

# --- 6. Data Transforms ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f"✅ Transforms defined (IMG_SIZE={IMG_SIZE})")

In [ ]:
# --- 7. Create Datasets & DataLoaders ---
# Split train into train + val (85/15)
full_train = hf_dataset['train']
total = len(full_train)
val_size = int(0.15 * total)
train_size = total - val_size

indices = torch.randperm(total).tolist()
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_subset = full_train.select(train_indices)
val_subset   = full_train.select(val_indices)
test_data    = hf_dataset['test']

train_dataset = FlowersDataset(train_subset, transform=train_transform)
val_dataset   = FlowersDataset(val_subset,   transform=val_transform)
test_dataset  = FlowersDataset(test_data,    transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"📂 Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"📦 Batches → Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# --- 8. Visualize Sample Images ---
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for display."""
    t = tensor.clone()
    for ch, m, s in zip(t, mean, std):
        ch.mul_(s).add_(m)
    return torch.clamp(t, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(FLOWER_NAMES[labels[i]], fontsize=9, fontweight='bold')
    ax.axis('off')
fig.suptitle("🌸 Sample Training Images — Oxford Flowers 102", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Class distribution
print("\n📊 Class Distribution:")
train_labels = [train_subset[i]['label'] for i in range(len(train_subset))]
label_counts = Counter(train_labels)

fig, ax = plt.subplots(figsize=(20, 4))
sorted_counts = sorted(label_counts.items())
ax.bar([x[0] for x in sorted_counts], [x[1] for x in sorted_counts],
       color='mediumpurple', alpha=0.85, edgecolor='indigo', linewidth=0.5)
ax.set_xlabel("Class Index")
ax.set_ylabel("Count")
ax.set_title("Training Set — Class Distribution")
plt.tight_layout()
plt.show()
print(f"Min samples/class: {min(label_counts.values())} | Max: {max(label_counts.values())}")

## 🏗️ Model — EfficientNet Transfer Learning

### Architecture Modifications:
```
EfficientNet-B0 (Pretrained ImageNet)
├── features (MBConv blocks + SE modules) → FROZEN in Phase 1
└── classifier (REPLACED)
    ├── Dropout(0.2)              ← original
    ├── Linear(1280 → 1000)       ← original (ImageNet)
    ↓  replaced with ↓
    ├── Linear(1280 → 512)
    ├── SiLU (Swish) + BatchNorm + Dropout(0.3)
    ├── Linear(512 → 256)
    ├── SiLU (Swish) + BatchNorm + Dropout(0.2)
    └── Linear(256 → 102)
```

### EfficientNet Key Innovations:
| Feature | Description |
|---------|-------------|
| **Compound Scaling** | Scale depth, width, resolution together with fixed ratio |
| **MBConv (Mobile Inverted Bottleneck)** | Expand → Depthwise Conv → SE → Project |
| **SiLU (Swish)** | `x · σ(x)` — smooth, non-monotonic activation |
| **SE Block** | Squeeze-and-Excitation for channel attention |
| **NAS (Neural Architecture Search)** | B0 baseline found via NAS, then scaled |
| **Stochastic Depth** | Drop paths during training for regularization |

In [ ]:
# --- 9. Build EfficientNet Model ---
def build_efficientnet(variant='b0', num_classes=102, dropout=0.3):
    """
    Load pretrained EfficientNet and replace classifier head.

    Supports variants: b0, b1, b2, b3, b4
    Uses SiLU (Swish) activation — EfficientNet's native activation function.
    """
    variant_map = {
        'b0': (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        'b1': (models.efficientnet_b1, models.EfficientNet_B1_Weights.IMAGENET1K_V1),
        'b2': (models.efficientnet_b2, models.EfficientNet_B2_Weights.IMAGENET1K_V1),
        'b3': (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1),
        'b4': (models.efficientnet_b4, models.EfficientNet_B4_Weights.IMAGENET1K_V1),
    }

    model_fn, weights = variant_map[variant]
    model = model_fn(weights=weights)

    # Get classifier input features (1280 for B0, scales up for B1-B4)
    in_features = model.classifier[1].in_features

    # Replace classifier with custom head using SiLU (Swish)
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, 512),
        nn.SiLU(inplace=True),         # Swish — EfficientNet's native activation
        nn.BatchNorm1d(512),
        nn.Dropout(p=dropout * 0.7),
        nn.Linear(512, 256),
        nn.SiLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(256, num_classes),
    )

    # Initialize new classifier weights (Kaiming for SiLU)
    for m in model.classifier.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    return model, in_features

model, classifier_in = build_efficientnet(variant=VARIANT, num_classes=NUM_CLASSES)
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📐 EfficientNet-{VARIANT.upper()}")
print(f"   Classifier in_features: {classifier_in}")
print(f"   Total params:     {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")
print(f"   Model size:       ~{total_params * 4 / 1024**2:.1f} MB (FP32)")
print(f"\n🔧 Classifier Head:")
print(model.classifier)

In [ ]:
# --- 10. Freeze Backbone for Phase 1 ---
def freeze_backbone(model):
    """Freeze all feature extraction layers."""
    for param in model.features.parameters():
        param.requires_grad = False
    print("🔒 Backbone FROZEN — training classifier head only")

def unfreeze_backbone(model):
    """Unfreeze all layers for fine-tuning."""
    for param in model.features.parameters():
        param.requires_grad = True
    print("🔓 Backbone UNFROZEN — fine-tuning all layers")

# Phase 1: Freeze backbone
freeze_backbone(model)

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total_params - trainable_now
print(f"   Trainable params (Phase 1): {trainable_now:,} / {total_params:,}")
print(f"   Frozen: {frozen:,} params ({frozen/total_params*100:.1f}%)")

## 🎯 Training — 2-Phase Strategy

### Phase 1: Feature Extraction (Backbone Frozen)
- Only classifier head trains → fast convergence, no catastrophic forgetting
- Higher learning rate (1e-3) with `ReduceLROnPlateau`

### Phase 2: Fine-Tuning (Full Model)
- Unfreeze backbone → differential learning rates
- Early features (`features[:5]`): `FINE_TUNE_LR × 0.1` (preserve low-level features)
- Late features (`features[5:]`): `FINE_TUNE_LR` (adapt high-level features)
- Classifier: `FINE_TUNE_LR × 5` (continue task adaptation)
- `CosineAnnealingWarmRestarts` scheduler for smooth LR decay

In [ ]:
# --- 11. Loss, Optimizer & Scheduler ---
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Phase 1 optimizer — only classifier params
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

print(f"⚙️  Loss:      CrossEntropyLoss (label_smoothing=0.1)")
print(f"⚙️  Optimizer: Adam (lr={LEARNING_RATE}, wd=1e-4)")
print(f"⚙️  Scheduler: ReduceLROnPlateau (patience=2, factor=0.5)")
print(f"⚙️  AMP:       {'Enabled ✅' if scaler else 'Disabled (CPU)'}")

In [ ]:
# --- 12. Training & Validation Functions ---
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with optional mixed precision."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate model on given loader."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

print("✅ Training & validation functions defined")

In [ ]:
# --- 13. Full Training Loop (2-Phase) ---
SAVE_PATH = f"best_efficientnet_{VARIANT}_flowers102.pth"

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'lr': [],         'phase': [],
}

best_val_acc = 0.0
total_epochs = FREEZE_EPOCHS + NUM_EPOCHS

print("=" * 70)
print(f"🚀 PHASE 1: Feature Extraction (Epochs 1-{FREEZE_EPOCHS})")
print(f"   Backbone FROZEN | LR = {LEARNING_RATE}")
print("=" * 70)

for epoch in range(1, total_epochs + 1):
    # --- Phase Transition ---
    if epoch == FREEZE_EPOCHS + 1:
        print("\n" + "=" * 70)
        print(f"🔥 PHASE 2: Fine-Tuning (Epochs {FREEZE_EPOCHS+1}-{total_epochs})")
        print(f"   Backbone UNFROZEN | Differential LR")
        print("=" * 70)

        unfreeze_backbone(model)

        # New optimizer with differential learning rates
        # EfficientNet features has 9 sequential blocks (0-8)
        num_feature_blocks = len(model.features)
        mid = num_feature_blocks // 2

        optimizer = optim.Adam([
            {'params': model.features[:mid].parameters(),    'lr': FINE_TUNE_LR * 0.1},  # Early layers
            {'params': model.features[mid:].parameters(),    'lr': FINE_TUNE_LR},         # Late layers
            {'params': model.classifier.parameters(),        'lr': FINE_TUNE_LR * 5},     # Classifier
        ], weight_decay=1e-4)

        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=2, eta_min=1e-7
        )

    # --- Train & Validate ---
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    # Scheduler step
    current_lr = optimizer.param_groups[-1]['lr']
    if epoch <= FREEZE_EPOCHS:
        scheduler.step(val_loss)
    else:
        scheduler.step(epoch - FREEZE_EPOCHS)

    # Record history
    phase = 1 if epoch <= FREEZE_EPOCHS else 2
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    history['phase'].append(phase)

    # Save best model
    improved = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'variant': VARIANT,
            'num_classes': NUM_CLASSES,
            'flower_names': FLOWER_NAMES,
        }, SAVE_PATH)
        improved = " ⭐ BEST"

    print(f"  [{epoch:2d}/{total_epochs}] "
          f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.2e} | ⏱ {elapsed:.1f}s{improved}")

print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")
print(f"💾 Model saved to: {SAVE_PATH}")

In [ ]:
# --- 14. Training History Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
epochs_range = range(1, len(history['train_loss']) + 1)
phase_boundary = FREEZE_EPOCHS + 0.5

# Loss curves
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
ax.plot(epochs_range, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('📉 Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy curves
ax = axes[1]
ax.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Train Acc')
ax.plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', markersize=3, label='Val Acc')
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('📈 Accuracy Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[2]
ax.plot(epochs_range, history['lr'], 'g-o', markersize=3)
ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('📐 Learning Rate Schedule')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle(f'EfficientNet-{VARIANT.upper()} Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 Evaluation — Test Set Performance

In [ ]:
# --- 15. Load Best Model & Test Evaluation ---
checkpoint = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} (Val Acc: {checkpoint['val_acc']:.4f})")

# Full test evaluation
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
        else:
            outputs = model(images)

        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f"\n🎯 Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Correct: {(all_preds == all_labels).sum()} / {len(all_labels)}")

# Top-5 accuracy
top5_correct = sum(1 for i in range(len(all_labels))
                   if all_labels[i] in np.argsort(all_probs[i])[-5:])
top5_acc = top5_correct / len(all_labels)
print(f"   Top-5 Accuracy: {top5_acc:.4f} ({top5_acc*100:.2f}%)")

In [ ]:
# --- 16. Classification Report ---
report = classification_report(all_labels, all_preds,
                               target_names=FLOWER_NAMES,
                               output_dict=True)
print("📋 Classification Report (Top 20 classes by F1-score):\n")

# Sort by F1 and show top 20
class_metrics = {k: v for k, v in report.items()
                 if k not in ['accuracy', 'macro avg', 'weighted avg']}
sorted_classes = sorted(class_metrics.items(), key=lambda x: x[1]['f1-score'], reverse=True)

print(f"{'Class':<30} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 75)
for name, metrics in sorted_classes[:20]:
    print(f"{name:<30} {metrics['precision']:>10.3f} {metrics['recall']:>10.3f} "
          f"{metrics['f1-score']:>10.3f} {int(metrics['support']):>10}")
print("-" * 75)
print(f"{'Macro Avg':<30} {report['macro avg']['precision']:>10.3f} "
      f"{report['macro avg']['recall']:>10.3f} {report['macro avg']['f1-score']:>10.3f}")
print(f"{'Weighted Avg':<30} {report['weighted avg']['precision']:>10.3f} "
      f"{report['weighted avg']['recall']:>10.3f} {report['weighted avg']['f1-score']:>10.3f}")

In [ ]:
# --- 17. Confusion Matrix (Top 20 most common classes) ---
top_k = 20
class_counts = Counter(all_labels)
top_classes = [cls for cls, _ in class_counts.most_common(top_k)]
top_names = [FLOWER_NAMES[c] for c in top_classes]

# Filter to only top-K classes
mask = np.isin(all_labels, top_classes)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

# Remap to 0..top_k-1
label_map = {old: new for new, old in enumerate(top_classes)}
mapped_labels = np.array([label_map[l] for l in filtered_labels])
mapped_preds = np.array([label_map.get(p, -1) for p in filtered_preds])

valid_mask = mapped_preds >= 0
cm = confusion_matrix(mapped_labels[valid_mask], mapped_preds[valid_mask], labels=range(top_k))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='BuPu',
            xticklabels=top_names, yticklabels=top_names, ax=ax,
            linewidths=0.5, square=True)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Top {top_k} Classes) — EfficientNet-{VARIANT.upper()}',
             fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 18. Per-Class Accuracy (Bottom & Top 10) ---
per_class_acc = {}
for cls_idx in range(NUM_CLASSES):
    mask = all_labels == cls_idx
    if mask.sum() > 0:
        per_class_acc[FLOWER_NAMES[cls_idx]] = (all_preds[mask] == cls_idx).mean()

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Bottom 10 (hardest classes)
ax = axes[0]
bottom_10 = sorted_acc[:10]
names = [x[0] for x in bottom_10]
accs = [x[1] * 100 for x in bottom_10]
bars = ax.barh(names, accs, color='#e74c3c', alpha=0.8, edgecolor='darkred')
ax.set_xlabel('Accuracy (%)')
ax.set_title('❌ Bottom 10 — Hardest Classes', fontweight='bold')
ax.set_xlim(0, 100)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)

# Top 10 (easiest classes)
ax = axes[1]
top_10 = sorted_acc[-10:]
names = [x[0] for x in top_10]
accs = [x[1] * 100 for x in top_10]
bars = ax.barh(names, accs, color='#2ecc71', alpha=0.8, edgecolor='darkgreen')
ax.set_xlabel('Accuracy (%)')
ax.set_title('✅ Top 10 — Easiest Classes', fontweight='bold')
ax.set_xlim(0, 105)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)

plt.suptitle(f'Per-Class Accuracy — EfficientNet-{VARIANT.upper()}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 19. Prediction Visualization ---
def show_predictions(model, dataset, class_names, device, n=16, ncols=4):
    """Display random predictions with confidence and ✅/❌ markers."""
    model.eval()
    indices = random.sample(range(len(dataset)), n)
    nrows = n // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))

    for i, ax in enumerate(axes.flat):
        image, label = dataset[indices[i]]
        img_display = denormalize(image).permute(1, 2, 0).numpy()

        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(device)
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)[0]
            pred_class = probs.argmax().item()
            confidence = probs[pred_class].item()

        # Top-3 predictions
        top3_probs, top3_idx = probs.topk(3)

        correct = pred_class == label
        color = 'green' if correct else 'red'
        symbol = '✅' if correct else '❌'

        ax.imshow(img_display)
        ax.set_title(f"{symbol} Pred: {class_names[pred_class]}\n"
                     f"True: {class_names[label]} ({confidence:.1%})",
                     fontsize=8, color=color, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'🔍 Random Test Predictions — EfficientNet-{VARIANT.upper()}',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, FLOWER_NAMES, device, n=16)

## 🔬 Single Image Inference & EfficientNet Family Comparison

In [ ]:
# --- 20. Single Image Inference Function ---
def predict_flower(model, image_path_or_pil, class_names, device, transform=None, top_k=5):
    """
    Predict flower species from a single image.

    Args:
        model: Trained EfficientNet model
        image_path_or_pil: str path or PIL Image
        class_names: list of class names
        device: torch device
        transform: preprocessing transform (defaults to val_transform)
        top_k: number of top predictions to show

    Returns:
        dict with prediction, confidence, top_k list
    """
    if transform is None:
        transform = val_transform

    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert('RGB')
    else:
        image = image_path_or_pil.convert('RGB')

    input_tensor = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]

    top_probs, top_indices = probs.topk(top_k)
    top_probs = top_probs.cpu().numpy()
    top_indices = top_indices.cpu().numpy()
    top_names = [class_names[i] for i in top_indices]

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                              gridspec_kw={'width_ratios': [1, 1.5]})

    axes[0].imshow(image)
    axes[0].set_title(f"🌸 Predicted: {top_names[0]} ({top_probs[0]:.1%})",
                      fontsize=12, fontweight='bold')
    axes[0].axis('off')

    colors = ['#8e44ad' if i == 0 else '#3498db' for i in range(top_k)]
    bars = axes[1].barh(range(top_k-1, -1, -1), top_probs, color=colors, edgecolor='white')
    axes[1].set_yticks(range(top_k-1, -1, -1))
    axes[1].set_yticklabels(top_names, fontsize=10)
    axes[1].set_xlabel('Confidence')
    axes[1].set_title(f'Top-{top_k} Predictions', fontweight='bold')
    axes[1].set_xlim(0, 1.1)
    for bar, prob in zip(bars, top_probs):
        axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f'{prob:.2%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    return {
        'prediction': top_names[0],
        'confidence': float(top_probs[0]),
        'top_k': list(zip(top_names, top_probs.tolist())),
    }

# Demo: predict a random test image
sample_idx = random.randint(0, len(hf_dataset['test']) - 1)
sample_image = hf_dataset['test'][sample_idx]['image']
sample_label = hf_dataset['test'][sample_idx]['label']

print(f"Ground Truth: {FLOWER_NAMES[sample_label]}")
result = predict_flower(model, sample_image, FLOWER_NAMES, device)
print(f"\nPrediction:   {result['prediction']} ({result['confidence']:.2%})")

In [ ]:
# --- 21. EfficientNet Variant Comparison (B0–B4) ---
print("📊 EfficientNet Family Comparison (B0 → B4)")
print("=" * 85)

# Reference ImageNet Top-1 accuracy (from PyTorch docs)
imagenet_acc = {'B0': 77.1, 'B1': 78.6, 'B2': 80.1, 'B3': 82.0, 'B4': 83.4}
input_res    = {'B0': 224,  'B1': 240,  'B2': 260,  'B3': 300,  'B4': 380}

variant_models = [
    ('B0', models.efficientnet_b0),
    ('B1', models.efficientnet_b1),
    ('B2', models.efficientnet_b2),
    ('B3', models.efficientnet_b3),
    ('B4', models.efficientnet_b4),
]

comparison_data = []
for name, ModelFn in variant_models:
    m = ModelFn(weights=None)
    params = sum(p.numel() for p in m.parameters())
    size_mb = params * 4 / 1024**2
    res = input_res[name]

    # Measure CPU inference latency
    dummy = torch.randn(1, 3, res, res)
    m.eval()
    times = []
    with torch.no_grad():
        for _ in range(20):
            t0 = time.time()
            _ = m(dummy)
            times.append(time.time() - t0)
    avg_ms = np.mean(times[5:]) * 1000

    comparison_data.append({
        'Variant': f'EfficientNet-{name}',
        'Parameters': f'{params/1e6:.1f}M',
        'Size (FP32)': f'{size_mb:.1f} MB',
        'CPU Latency': f'{avg_ms:.1f} ms',
        'Resolution': f'{res}×{res}',
        'ImageNet Top-1': f'{imagenet_acc[name]:.1f}%',
        'params_raw': params,
        'res_raw': res,
        'acc_raw': imagenet_acc[name],
        'latency_raw': avg_ms,
    })
    del m

# Display table
header = f"{'Variant':<20} {'Params':>10} {'Size':>10} {'Latency':>12} {'Resolution':>12} {'Top-1':>10}"
print(header)
print("-" * 78)
for d in comparison_data:
    print(f"{d['Variant']:<20} {d['Parameters']:>10} {d['Size (FP32)']:>10} "
          f"{d['CPU Latency']:>12} {d['Resolution']:>12} {d['ImageNet Top-1']:>10}")

print("\n📌 Compound Scaling Trade-offs:")
print(f"   • B0→B4: Params {comparison_data[0]['Parameters']} → {comparison_data[-1]['Parameters']} "
      f"({comparison_data[-1]['params_raw']/comparison_data[0]['params_raw']:.1f}× increase)")
print(f"   • B0→B4: Accuracy {imagenet_acc['B0']}% → {imagenet_acc['B4']}% "
      f"(+{imagenet_acc['B4']-imagenet_acc['B0']:.1f}%)")
print(f"   • B0→B4: Latency {comparison_data[0]['CPU Latency']} → {comparison_data[-1]['CPU Latency']}")
print(f"   • Each step scales depth×width×resolution uniformly")

In [ ]:
# --- 22. Compound Scaling Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

variants = [d['Variant'].split('-')[1] for d in comparison_data]
params_m = [d['params_raw'] / 1e6 for d in comparison_data]
resolutions = [d['res_raw'] for d in comparison_data]
accuracies = [d['acc_raw'] for d in comparison_data]

# Plot 1: Parameters vs Variant
ax = axes[0]
colors_bar = ['#8e44ad', '#9b59b6', '#a569bd', '#bb8fce', '#d2b4de']
bars = ax.bar(variants, params_m, color=colors_bar, edgecolor='indigo', alpha=0.9)
ax.set_xlabel('Variant')
ax.set_ylabel('Parameters (M)')
ax.set_title('📊 Model Size by Variant', fontweight='bold')
for bar, p in zip(bars, params_m):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{p:.1f}M', ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Resolution vs Variant
ax = axes[1]
ax.plot(variants, resolutions, 'o-', color='#e74c3c', linewidth=2, markersize=10)
ax.fill_between(range(len(variants)), resolutions, alpha=0.15, color='#e74c3c')
ax.set_xlabel('Variant')
ax.set_ylabel('Input Resolution')
ax.set_title('📐 Resolution Scaling', fontweight='bold')
for i, (v, r) in enumerate(zip(variants, resolutions)):
    ax.annotate(f'{r}×{r}', (i, r), textcoords="offset points",
                xytext=(0, 12), ha='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Params vs Accuracy (scatter)
ax = axes[2]
scatter = ax.scatter(params_m, accuracies, s=200, c=colors_bar,
                     edgecolors='indigo', linewidths=2, zorder=5)
ax.plot(params_m, accuracies, '--', color='gray', alpha=0.5, zorder=1)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('ImageNet Top-1 Acc (%)')
ax.set_title('🎯 Efficiency Frontier', fontweight='bold')
for p, a, v in zip(params_m, accuracies, variants):
    ax.annotate(v, (p, a), textcoords="offset points",
                xytext=(10, 5), fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

fig.suptitle('EfficientNet Compound Scaling: depth=α^φ, width=β^φ, resolution=γ^φ\n'
             'α=1.2, β=1.1, γ=1.15  |  α·β²·γ² ≈ 2',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 23. Export to ONNX (Deployment Ready) ---
print("📱 Exporting to ONNX for Deployment...")

model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
onnx_path = f"efficientnet_{VARIANT}_flowers102.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size'},
        'output': {0: 'batch_size'},
    },
)

onnx_size = os.path.getsize(onnx_path) / 1024**2
pytorch_size = os.path.getsize(SAVE_PATH) / 1024**2

print(f"✅ ONNX model exported: {onnx_path}")
print(f"   ONNX size:    {onnx_size:.1f} MB")
print(f"   PyTorch size: {pytorch_size:.1f} MB")
print(f"\n💡 Deployment Options:")
print(f"   • Android/iOS: ONNX Runtime Mobile or TFLite (via onnx-tf)")
print(f"   • Edge/IoT:    ONNX Runtime + INT8 quantization")
print(f"   • Web:         ONNX Runtime Web (WebAssembly)")
print(f"   • Server:      TorchServe or Triton Inference Server")
print(f"\n📌 EfficientNet Edge Advantage:")
print(f"   • Compound-scaled B0 gives best accuracy-per-FLOP")
print(f"   • MBConv blocks are depthwise separable → hardware efficient")
print(f"   • SE blocks add minimal overhead (<10% extra compute)")

## 📝 Summary

### Results
| Metric | Value |
|--------|-------|
| **Model** | EfficientNet-B0 (Transfer Learning) |
| **Dataset** | Oxford Flowers 102 (102 classes) |
| **Training** | 2-Phase (Frozen → Fine-tune) |
| **Best Val Accuracy** | Check training logs above |
| **ONNX Export** | ✅ Deployment ready |

### EfficientNet vs Other Architectures

| Model | Params | ImageNet Top-1 | Key Innovation |
|-------|--------|----------------|----------------|
| **EfficientNet-B0** | 5.3M | 77.1% | Compound scaling (NAS baseline) |
| **EfficientNet-B4** | 19M | 83.4% | Scaled B0 with φ=2.0 |
| **MobileNetV3-Large** | 5.4M | 75.2% | h-swish + SE + NAS |
| **ResNet-50** | 25.6M | 76.1% | Skip connections |
| **ConvNeXt-Tiny** | 28.6M | 82.1% | Modernized ConvNet |

### EfficientNet Key Takeaways
1. **Compound Scaling** — Scale depth, width, resolution together with fixed ratio (α·β²·γ²≈2)
2. **SiLU (Swish)** — `x·σ(x)` smooth activation outperforms ReLU
3. **MBConv + SE** — Inverted residuals with squeeze-excitation for efficiency
4. **NAS Baseline** — B0 architecture found via Neural Architecture Search
5. **Scaling Law** — Each φ step ≈ doubles compute, consistent accuracy gains
6. **Best Accuracy/FLOP** — More efficient than scaling width-only or depth-only
7. **Transfer Learning** — 2-phase strategy works well across all variants

### Compound Scaling Formula
$$\text{depth} = \alpha^\phi, \quad \text{width} = \beta^\phi, \quad \text{resolution} = \gamma^\phi$$
$$\alpha = 1.2, \quad \beta = 1.1, \quad \gamma = 1.15$$
$$\text{subject to} \quad \alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$$

### References
- [EfficientNet: Rethinking Model Scaling for CNNs (Tan & Le, 2019)](https://arxiv.org/abs/1905.11946)
- [EfficientNetV2: Smaller Models and Faster Training (Tan & Le, 2021)](https://arxiv.org/abs/2104.00298)
- [PyTorch EfficientNet Docs](https://pytorch.org/vision/stable/models/efficientnet.html)
- [Oxford Flowers 102 Dataset](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/)